# DK-SOFNN: Data-Knowledge-Driven Self-Organizing Fuzzy Neural Network
## Combined Cycle Power Plant (CCPP) Dataset

**Paper:** Han, H., Li, Z., Qiao, J. (2023). *Data-Knowledge-Driven Self-Organizing Fuzzy Neural Network.*  
IEEE Transactions on Fuzzy Systems.

### Notebook Overview
1. Data loading & preprocessing (Source: 1500, Target train: 100, Target test: 300)
2. DK-SOFNN algorithm implementation (Eqs. 7–30)
3. Source FNN trained with **K₀ = 10** and **K₀ = 20** (Fig 11)
4. Target FNN trained starting from **K₀ = 15** (Fig 12)
5. Comparison methods: GM-FTL, RS-SOFNN, APSO-FNN
6. Figures 11–17 (matching paper Results screenshots)
7. Table III equivalent (30-run mean ± std)
8. All fuzzy rules printed: initial → added/pruned → final

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────
!pip install openpyxl -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MaxNLocator
from copy import deepcopy
import warnings, os, pickle, time
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'lines.linewidth': 1.5,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
})

EPS     = 1e-8
EPS_IDX = 1e-10

print('Setup complete.')

In [ ]:
# ── Cell 2: Data Loading & Preprocessing ─────────────────────────
# Paper Section IV-A-2: CCPP dataset
# Source: 1500 samples | Target train: 100 | Target test: 300
# Domain shift: noise N(50,10) added to target TRAIN outputs (paper)
# Evaluation on TRUE (clean) test labels

def load_ccpp(filepath='Folds5x2_pp.xlsx'):
    try:
        df = pd.read_excel(filepath, sheet_name='Sheet1')
    except Exception:
        df = pd.read_excel(filepath)
    print(f'Dataset loaded: {df.shape[0]} samples, {df.shape[1]} columns')
    print(f'Columns: {df.columns.tolist()}')
    print('\nStatistics:')
    print(df.describe().round(2))
    return df.values.astype(np.float64)


def prepare_ccpp(data, n_src=1500, n_tgt_tr=100, n_tgt_te=300,
                 noise_mu=50, noise_sigma=10, seed=42):
    np.random.seed(seed)
    idx  = np.random.permutation(len(data))
    data = data[idx]
    X, y = data[:, :-1], data[:, -1]

    Xs  = X[:n_src];                ys  = y[:n_src]
    Xtt = X[n_src:n_src+n_tgt_tr]
    ytt = y[n_src:n_src+n_tgt_tr] + np.random.normal(noise_mu, noise_sigma, n_tgt_tr)
    Xte = X[n_src+n_tgt_tr:n_src+n_tgt_tr+n_tgt_te]
    yte = y[n_src+n_tgt_tr:n_src+n_tgt_tr+n_tgt_te]   # clean labels

    Xmin, Xmax = Xs.min(0), Xs.max(0)
    def norm_X(Z): return (Z - Xmin) / (Xmax - Xmin + 1e-8)

    ymin, ymax = ys.min(), ys.max()
    def norm_y(v): return (v - ymin) / (ymax - ymin + 1e-8)

    return {
        'Xs':  norm_X(Xs),  'ys':  norm_y(ys),
        'Xtt': norm_X(Xtt), 'ytt': norm_y(ytt),
        'Xte': norm_X(Xte), 'yte': norm_y(yte),
        'ymin': ymin, 'ymax': ymax
    }


raw  = load_ccpp('Folds5x2_pp.xlsx')
DS   = prepare_ccpp(raw)
print(f'\nSplit → Source: {DS["Xs"].shape[0]}  '
      f'Target-train: {DS["Xtt"].shape[0]}  '
      f'Target-test: {DS["Xte"].shape[0]}')

In [ ]:
# ── Cell 3: RBF Fuzzy Neural Network ─────────────────────────────
# Architecture: x(P) → Gaussian RBF layer (K rules) → y
# Implements Eqs. 7 (source) and 16 (target)

class RBFFNN:
    def __init__(self, n_in, n_rules, lr=0.1):
        self.P  = n_in
        self.K  = n_rules
        self.lr = lr
        self.C  = np.random.rand(n_rules, n_in)
        self.S  = np.full((n_rules, n_in), 0.3)
        self.W  = np.random.randn(n_rules) * 0.1

    # ── Forward pass (Eq. 7 / 16) ──────────────────────────────
    def forward(self, x):
        diff = x[None, :] - self.C
        u    = np.exp(-0.5 * ((diff / (self.S + EPS)) ** 2).sum(1))
        v    = u / (u.sum() + EPS_IDX)
        y    = float(self.W @ v)
        return y, u, v

    def predict(self, X):
        return np.array([self.forward(x)[0] for x in X])

    # ── Parameter update (Eqs. 8-9, 24-27) ────────────────────
    def update(self, x, y, yd, u, v,
               alpha=1.0, beta=0.0, Ws=None, Cs=None, Ss=None):
        e  = y - yd
        gW = alpha * e * v
        if beta > 0 and Ws is not None:
            n = min(self.K, len(Ws))
            gW[:n] += beta * (self.W[:n] - Ws[:n])

        gC = np.zeros_like(self.C)
        gS = np.zeros_like(self.S)
        for b in range(self.K):
            s2 = self.S[b] ** 2 + EPS
            s3 = self.S[b] ** 3 + EPS
            Dc = v[b] * (self.W[b] - y) * (x - self.C[b]) / s2
            Ds = v[b] * (self.W[b] - y) * (x - self.C[b]) ** 2 / s3
            gC[b] = alpha * e * Dc
            gS[b] = alpha * e * Ds
            if beta > 0 and Cs is not None:
                kb = min(b, len(Cs) - 1)
                gC[b] += beta * (self.C[b] - Cs[kb])
                gS[b] += beta * (self.S[b] - Ss[kb])

        self.W -= self.lr * gW
        self.C -= self.lr * gC
        self.S  = np.maximum(self.S - self.lr * gS, 0.01)

    # ── Structure operations ────────────────────────────────────
    def add_rule(self, x, l):
        new_c = x.copy()
        new_s = np.abs(x - self.C[l]) / 2.0 + 0.1
        self.C = np.vstack([self.C, new_c])
        self.S = np.vstack([self.S, new_s])
        self.W = np.append(self.W, self.W[l])
        self.K += 1

    def del_rule(self, l):
        if self.K <= 2:
            return
        self.C = np.delete(self.C, l, 0)
        self.S = np.delete(self.S, l, 0)
        self.W = np.delete(self.W, l)
        self.K -= 1

    def replace_rule(self, l_tgt, c_src, s_src, w_src):
        self.C[l_tgt] = c_src
        self.S[l_tgt] = s_src
        self.W[l_tgt] = w_src

    def clone(self):
        f   = RBFFNN(self.P, self.K, self.lr)
        f.C = self.C.copy()
        f.S = self.S.copy()
        f.W = self.W.copy()
        return f


print('RBFFNN defined.')

In [ ]:
# ── Cell 4: Fuzzy Rule Quality Indices (Eqs. 10-12) ──────────────
# R: Similarity (redundancy)  — higher R = more redundant
# M: Sensitivity              — higher M = fires more
# C: Contribution             — higher C = more important

def compute_indices(U, y_arr):
    N, K  = U.shape
    eps   = EPS_IDX
    if N < 3:
        return np.ones(K), np.ones(K)/K, np.ones(K)/K

    total_u = U.sum(1)

    # Similarity R (Eq. 10)
    R = np.zeros(K)
    for l in range(K):
        a = U[:, l] - U[:, l].mean()
        b = total_u  - total_u.mean()
        denom = np.linalg.norm(a) * np.linalg.norm(b) + eps
        R[l]  = 1.0 / (abs(float(a @ b) / denom) + eps)

    # Sensitivity M (Eq. 11)
    M = U.sum(0) / (U.sum() + eps)

    # Contribution C (Eq. 12)
    C = np.zeros(K)
    for l in range(K):
        a = U[:, l] - y_arr
        b = total_u  - y_arr
        denom = np.linalg.norm(a) * np.linalg.norm(b) + eps
        d_l   = float(a @ b) / denom
        C[l]  = 1.0 / (abs(d_l) + eps)

    return R, M, C


print('Index functions defined.')

In [ ]:
# ── Cell 5: Source FNN Training (Eqs. 8-15) ──────────────────────

def train_source_fnn(Xs, ys, K0=20, n_iter=300, lr=0.1,
                     N_win=100, K_max=30, K_min=3, seed=42):
    np.random.seed(seed)
    P   = Xs.shape[1]
    fnn = RBFFNN(P, K0, lr)
    idx = np.random.choice(len(Xs), K0, replace=False)
    fnn.C = Xs[idx].copy()

    # Capture initial rules snapshot
    initial_rules = [{'rule': k+1,
                      'C': fnn.C[k].copy(),
                      'S': fnn.S[k].copy(),
                      'W': float(fnn.W[k])}
                     for k in range(K0)]

    rule_hist      = [K0]
    rmse_hist      = []
    pruned_log     = []
    added_log      = []

    U_buf    = np.zeros((N_win, K0))
    y_buf    = np.zeros(N_win)
    t_global = 0

    for ep in range(n_iter):
        sq_errs = []
        for x, yd in zip(Xs, ys):
            y, u, v = fnn.forward(x)
            sq_errs.append((y - yd)**2)

            ti = t_global % N_win
            if U_buf.shape[1] != fnn.K:
                U_buf = np.zeros((N_win, fnn.K))
            U_buf[ti] = u
            y_buf[ti] = y
            t_global += 1

            fnn.update(x, y, yd, u, v)

            if t_global >= N_win and t_global % N_win == 0:
                R, M, C = compute_indices(U_buf, y_buf)
                gi = int(np.argmin(R))
                pi = int(np.argmax(R))

                # Growing (Eq. 13)
                if (gi == int(np.argmax(M)) == int(np.argmax(C)) and fnn.K < K_max):
                    added_log.append({'rule_idx': fnn.K, 'epoch': ep+1,
                                      'C': x.copy(), 'W': float(fnn.W[gi])})
                    fnn.add_rule(x, gi)
                    U_buf = np.zeros((N_win, fnn.K))
                    t_global = 0

                # Pruning (Eq. 15)
                elif (pi == int(np.argmin(M)) == int(np.argmin(C)) and fnn.K > K_min):
                    pruned_log.append({'rule_idx': pi+1, 'epoch': ep+1,
                                       'C': fnn.C[pi].copy(),
                                       'S': fnn.S[pi].copy(),
                                       'W': float(fnn.W[pi])})
                    fnn.del_rule(pi)
                    U_buf = np.zeros((N_win, fnn.K))
                    t_global = 0

        rmse_hist.append(np.sqrt(np.mean(sq_errs)))
        rule_hist.append(fnn.K)

    return fnn, rule_hist, rmse_hist, pruned_log, added_log, initial_rules


print('Source FNN training function defined.')

In [ ]:
# ── Cell 6: DK-SOFNN Target Training (Eqs. 17-30) ────────────────

def train_dk_sofnn(Xtt, ytt, src_fnn, Xs, ys,
                   K0_tgt=15, n_iter=300, lr=0.01,
                   N_win=10, H=10, K_max=25, K_min=2, seed=42):
    """
    K0_tgt: initial rule count for target FNN (paper uses 15)
    """
    np.random.seed(seed)
    P = Xtt.shape[1]

    # Initialise target FNN from source (transfer structure & params)
    fnn   = src_fnn.clone()
    fnn.lr = lr

    # Adjust to K0_tgt rules
    while fnn.K < K0_tgt:
        l = np.random.randint(fnn.K)
        fnn.add_rule(Xtt[np.random.randint(len(Xtt))], l)
    while fnn.K > K0_tgt and fnn.K > K_min:
        fnn.del_rule(fnn.K - 1)

    # Reference indices on SOURCE domain
    U_src    = np.array([src_fnn.forward(x)[1] for x in Xs])
    y_src    = np.array([src_fnn.forward(x)[0] for x in Xs])
    R_s, M_s, C_s = compute_indices(U_src, y_src)
    Rbar_s, Mbar_s, Cbar_s = R_s.mean(), M_s.mean(), C_s.mean()

    # H alpha/beta pairs (Eq. 29)
    alphas = np.linspace(0.50, 1.00, H)
    betas  = 1.0 - alphas

    rule_hist = [fnn.K]
    rmse_hist = []
    U_buf     = np.zeros((N_win, fnn.K))
    y_buf     = np.zeros(N_win)
    t_global  = 0

    for ep in range(n_iter):
        sq_errs = []
        for t, (x, yd) in enumerate(zip(Xtt, ytt)):
            y, u, v = fnn.forward(x)
            sq_errs.append((y - yd)**2)

            ti = t_global % N_win
            if U_buf.shape[1] != fnn.K:
                U_buf = np.zeros((N_win, fnn.K))
            U_buf[ti] = u
            y_buf[ti] = y
            t_global += 1

            # Align source params to current target size
            Ks, Kt = src_fnn.K, fnn.K
            n       = min(Ks, Kt)
            Ws      = np.zeros(Kt);           Ws[:n] = src_fnn.W[:n]
            Cs      = np.zeros((Kt, P));      Cs[:n] = src_fnn.C[:n]
            Ss      = np.full((Kt, P), 0.3);  Ss[:n] = src_fnn.S[:n]

            # Parameter Reinforcement (Eqs. 24-30)
            W0, C0, S0 = fnn.W.copy(), fnn.C.copy(), fnn.S.copy()
            bW, bC, bS = W0, C0, S0
            best_score = np.inf

            for h in range(H):
                fnn.W, fnn.C, fnn.S = W0.copy(), C0.copy(), S0.copy()
                fnn.update(x, y, yd, u, v,
                           alpha=alphas[h], beta=betas[h],
                           Ws=Ws, Cs=Cs, Ss=Ss)
                if t + 1 < len(Xtt):
                    y_next, _, _ = fnn.forward(Xtt[t+1])
                    score = (y_next - ytt[t+1])**2
                else:
                    score = (fnn.forward(x)[0] - yd)**2

                if score < best_score:
                    best_score = score
                    bW, bC, bS = fnn.W.copy(), fnn.C.copy(), fnn.S.copy()

            fnn.W, fnn.C, fnn.S = bW, bC, bS

            # Structure Compensation (Eqs. 17-22) every N_win steps
            if t_global >= N_win and t_global % N_win == 0:
                R_t, M_t, C_t   = compute_indices(U_buf, y_buf)
                Rbar_t = R_t.mean()
                Mbar_t = M_t.mean()
                Cbar_t = C_t.mean()

                # Growing (Eq. 17)
                if (Rbar_t <= Rbar_s and Mbar_t >= Mbar_s and
                        Cbar_t <= Cbar_s and fnn.K < K_max):
                    l = int(np.argmax(C_t))
                    fnn.add_rule(x, l)
                    U_buf = np.zeros((N_win, fnn.K))
                    t_global = 0

                # Pruning (Eq. 19)
                elif fnn.K > K_min:
                    prune_l = -1
                    cond_A  = (Rbar_t >= Rbar_s and Cbar_t >= Cbar_s)
                    cond_B  = (Mbar_t <= Mbar_s and Cbar_t >= Cbar_s)

                    if cond_A and cond_B:
                        prune_l = int(np.argmax(R_t - M_t - C_t))
                    elif cond_A:
                        prune_l = int(np.argmin(M_t)) if Mbar_t <= Mbar_s \
                                  else int(np.argmax(R_t))
                    elif cond_B:
                        prune_l = int(np.argmin(M_t)) if Rbar_t <= Rbar_s \
                                  else int(np.argmax(R_t - M_t))

                    if prune_l >= 0:
                        fnn.del_rule(prune_l)
                        U_buf = np.zeros((N_win, fnn.K))
                        t_global = 0

                # Compensating (Eq. 20)
                elif ((Rbar_t >= Rbar_s or Mbar_t <= Mbar_s) and
                      Cbar_t <= Cbar_s and src_fnn.K > 0):
                    score_t = -R_t + M_t + C_t
                    l_worst = int(np.argmin(score_t))
                    z_best  = int(np.argmax(np.abs(src_fnn.W)))
                    fnn.replace_rule(l_worst,
                                     src_fnn.C[z_best],
                                     src_fnn.S[z_best],
                                     src_fnn.W[z_best])

        rmse_hist.append(np.sqrt(np.mean(sq_errs)))
        rule_hist.append(fnn.K)

    return fnn, rule_hist, rmse_hist


print('DK-SOFNN training function defined.')

In [ ]:
# ── Cell 7: Comparison Methods ────────────────────────────────────

# ---- GM-FTL ----
def train_gm_ftl(Xtt, ytt, src_fnn, n_iter=300, lr=0.01, seed=42):
    np.random.seed(seed)
    fnn = src_fnn.clone()
    fnn.lr = lr
    Ws, Cs, Ss = src_fnn.W.copy(), src_fnn.C.copy(), src_fnn.S.copy()
    rmse_hist = []
    for _ in range(n_iter):
        sq = []
        for x, yd in zip(Xtt, ytt):
            y, u, v = fnn.forward(x)
            sq.append((y - yd)**2)
            fnn.update(x, y, yd, u, v, alpha=0.9, beta=0.1,
                       Ws=Ws, Cs=Cs, Ss=Ss)
        rmse_hist.append(np.sqrt(np.mean(sq)))
    return fnn, rmse_hist


# ---- RS-SOFNN ----
def train_rs_sofnn(Xtt, ytt, K0=20, n_iter=300, lr=0.01,
                   N_win=10, K_max=25, K_min=2, seed=42):
    np.random.seed(seed)
    P   = Xtt.shape[1]
    fnn = RBFFNN(P, K0, lr)
    idx = np.random.choice(len(Xtt), min(K0, len(Xtt)), replace=False)
    fnn.C = Xtt[idx].copy()
    U_buf = np.zeros((N_win, K0))
    y_buf = np.zeros(N_win)
    t_g   = 0
    rmse_hist = []
    for _ in range(n_iter):
        sq = []
        for x, yd in zip(Xtt, ytt):
            y, u, v = fnn.forward(x)
            sq.append((y - yd)**2)
            ti = t_g % N_win
            if U_buf.shape[1] != fnn.K:
                U_buf = np.zeros((N_win, fnn.K))
            U_buf[ti] = u; y_buf[ti] = y; t_g += 1
            fnn.update(x, y, yd, u, v)
            if t_g >= N_win and t_g % N_win == 0:
                M   = U_buf.sum(0) / (U_buf.sum() + EPS_IDX)
                thr = 1.0 / fnn.K
                if fnn.K < K_max and M.min() < thr * 0.3:
                    fnn.add_rule(x, int(np.argmax(M)))
                    U_buf = np.zeros((N_win, fnn.K)); t_g = 0
                elif fnn.K > K_min and M.max() < thr * 1.5:
                    fnn.del_rule(int(np.argmin(M)))
                    U_buf = np.zeros((N_win, fnn.K)); t_g = 0
        rmse_hist.append(np.sqrt(np.mean(sq)))
    return fnn, rmse_hist


# ---- APSO-FNN ----
def train_apso_fnn(Xtt, ytt, K0=20, n_iter=300, lr=0.01,
                   N_win=10, K_max=25, K_min=2, seed=42):
    np.random.seed(seed)
    P   = Xtt.shape[1]
    fnn = RBFFNN(P, K0, lr)
    idx = np.random.choice(len(Xtt), min(K0, len(Xtt)), replace=False)
    fnn.C = Xtt[idx].copy()
    U_buf = np.zeros((N_win, K0))
    y_buf = np.zeros(N_win)
    t_g   = 0
    rmse_hist = []
    for _ in range(n_iter):
        sq = []
        for x, yd in zip(Xtt, ytt):
            y, u, v = fnn.forward(x)
            sq.append((y - yd)**2)
            ti = t_g % N_win
            if U_buf.shape[1] != fnn.K:
                U_buf = np.zeros((N_win, fnn.K))
            U_buf[ti] = u; y_buf[ti] = y; t_g += 1
            fnn.update(x, y, yd, u, v, alpha=1.0, beta=0.0)
            if t_g >= N_win and t_g % N_win == 0:
                R, M, C = compute_indices(U_buf, y_buf)
                gi, pi  = int(np.argmin(R)), int(np.argmax(R))
                if gi == int(np.argmax(M)) and fnn.K < K_max:
                    fnn.add_rule(x, gi)
                    U_buf = np.zeros((N_win, fnn.K)); t_g = 0
                elif pi == int(np.argmin(M)) and fnn.K > K_min:
                    fnn.del_rule(pi)
                    U_buf = np.zeros((N_win, fnn.K)); t_g = 0
        rmse_hist.append(np.sqrt(np.mean(sq)))
    return fnn, rmse_hist


# ---- Fixed-structure FNN (for Figs 13-15) ----
def train_fixed_fnn(Xtt, ytt, src_fnn, K_fixed,
                    n_iter=300, lr=0.01, seed=42):
    np.random.seed(seed)
    P   = Xtt.shape[1]
    fnn = RBFFNN(P, K_fixed, lr)
    n   = min(K_fixed, src_fnn.K)
    fnn.C[:n] = src_fnn.C[:n]
    fnn.S[:n] = src_fnn.S[:n]
    fnn.W[:n] = src_fnn.W[:n]
    rmse_hist = []
    for _ in range(n_iter):
        sq = []
        for x, yd in zip(Xtt, ytt):
            y, u, v = fnn.forward(x)
            sq.append((y - yd)**2)
            fnn.update(x, y, yd, u, v)
        rmse_hist.append(np.sqrt(np.mean(sq)))
    return fnn, rmse_hist


print('Comparison methods defined.')

In [ ]:
# ── Cell 8: Evaluation Metrics (Eqs. 42-44) ──────────────────────

def rmse(y_pred, y_true):
    return float(np.sqrt(np.mean((y_pred - y_true)**2)))

def smape(y_pred, y_true):
    num = 2 * np.abs(y_pred - y_true)
    den = np.abs(y_pred) + np.abs(y_true) + EPS_IDX
    return float(np.mean(num / den))

def mase(y_pred, y_true):
    scale = np.mean(np.abs(y_true[1:] - y_true[:-1])) + EPS_IDX
    return float(np.mean(np.abs(y_pred - y_true)) / scale)


print('Metrics defined.')

In [ ]:
# ── Cell 9: Single-Run Training (for all graphs) ──────────────────
# Uses a fixed seed so rule histories are integers (no averaging)
# Source FNN: K0=10 and K0=20 (Fig 11)
# Target FNN: starts at K0=15 (Fig 12)
# Fixed-structure targets: K=11, K=13, K=15 (Figs 13-15)

SEED_SINGLE = 0

print('='*60)
print('Training SOURCE FNN  K0 = 10 ...')
src10, rule_s10, rmse_s10, pruned10, added10, init10 = train_source_fnn(
    DS['Xs'], DS['ys'], K0=10, n_iter=300, lr=0.1,
    N_win=100, seed=SEED_SINGLE)
print(f'  Converged to  K = {int(src10.K)} rules,  '
      f'Final RMSE = {rmse_s10[-1]:.4f}')

print('Training SOURCE FNN  K0 = 20 ...')
src20, rule_s20, rmse_s20, pruned20, added20, init20 = train_source_fnn(
    DS['Xs'], DS['ys'], K0=20, n_iter=300, lr=0.1,
    N_win=100, seed=SEED_SINGLE)
print(f'  Converged to  K = {int(src20.K)} rules,  '
      f'Final RMSE = {rmse_s20[-1]:.4f}')

print('Training DK-SOFNN TARGET  K0 = 15 ...')
dk_fnn, rule_tgt, rmse_tgt = train_dk_sofnn(
    DS['Xtt'], DS['ytt'], src20, DS['Xs'], DS['ys'],
    K0_tgt=15, n_iter=300, lr=0.01, N_win=10, H=10,
    seed=SEED_SINGLE)
print(f'  Converged to  K = {int(dk_fnn.K)} rules,  '
      f'Final train RMSE = {rmse_tgt[-1]:.4f}')

print('Training FIXED-STRUCTURE targets (K=11, 13, 15) ...')
fnn11, rmse_11 = train_fixed_fnn(DS['Xtt'], DS['ytt'], src20,
                                  K_fixed=11, n_iter=300, lr=0.01, seed=SEED_SINGLE)
fnn13, rmse_13 = train_fixed_fnn(DS['Xtt'], DS['ytt'], src20,
                                  K_fixed=13, n_iter=300, lr=0.01, seed=SEED_SINGLE)
fnn15, rmse_15 = train_fixed_fnn(DS['Xtt'], DS['ytt'], src20,
                                  K_fixed=15, n_iter=300, lr=0.01, seed=SEED_SINGLE)

# Test predictions (single run)
y_te     = DS['yte']
pred11   = fnn11.predict(DS['Xte'])
pred13   = fnn13.predict(DS['Xte'])
pred15   = fnn15.predict(DS['Xte'])
pred_dk  = dk_fnn.predict(DS['Xte'])

print('\nTest RMSE (single run, normalised):')
for lbl, p in [('K=11', pred11), ('K=13', pred13),
                ('K=15', pred15), ('DK-SOFNN', pred_dk)]:
    print(f'  {lbl:10s}: {rmse(p, y_te):.4f}')

In [ ]:
# ── Cell 10: 30-Run Experiment (for Table III) ────────────────────
# Produces mean ± std over 30 independent runs
# Saves best-run predictions for method comparison graphs

N_RUNS = int(os.environ.get('DK_SOFNN_N_RUNS', 30))

results = {m: {'K': [], 'RMSE': [], 'sMAPE': [], 'MASE': []}
           for m in ['DK-SOFNN', 'GM-FTL', 'RS-SOFNN', 'APSO-FNN']}

best_run_rmse = np.inf
best_preds    = {}

print(f'Running {N_RUNS} independent experiments ...')
hdr = f"{'Run':>4}  {'DK-SOFNN':>10}  {'GM-FTL':>10}  {'RS-SOFNN':>10}  {'APSO-FNN':>10}  {'Elapsed':>9}"
print(hdr)
print('-' * len(hdr))

t0 = time.time()
for run in range(N_RUNS):
    seed = run * 37

    src, _, _, _, _, _ = train_source_fnn(
        DS['Xs'], DS['ys'], K0=20, n_iter=300, lr=0.1, N_win=100, seed=seed)

    dk, _, _ = train_dk_sofnn(
        DS['Xtt'], DS['ytt'], src, DS['Xs'], DS['ys'],
        K0_tgt=15, n_iter=300, lr=0.01, N_win=10, H=10, seed=seed)
    p_dk  = dk.predict(DS['Xte'])
    r_dk  = rmse(p_dk, DS['yte'])
    results['DK-SOFNN']['K'].append(int(dk.K))
    results['DK-SOFNN']['RMSE'].append(r_dk)
    results['DK-SOFNN']['sMAPE'].append(smape(p_dk, DS['yte']))
    results['DK-SOFNN']['MASE'].append(mase(p_dk, DS['yte']))

    gm, _ = train_gm_ftl(DS['Xtt'], DS['ytt'], src, n_iter=300, lr=0.01, seed=seed)
    p_gm  = gm.predict(DS['Xte'])
    r_gm  = rmse(p_gm, DS['yte'])
    results['GM-FTL']['K'].append(int(gm.K))
    results['GM-FTL']['RMSE'].append(r_gm)
    results['GM-FTL']['sMAPE'].append(smape(p_gm, DS['yte']))
    results['GM-FTL']['MASE'].append(mase(p_gm, DS['yte']))

    rs, _ = train_rs_sofnn(DS['Xtt'], DS['ytt'], K0=20, n_iter=300, lr=0.01, N_win=10, seed=seed)
    p_rs  = rs.predict(DS['Xte'])
    r_rs  = rmse(p_rs, DS['yte'])
    results['RS-SOFNN']['K'].append(int(rs.K))
    results['RS-SOFNN']['RMSE'].append(r_rs)
    results['RS-SOFNN']['sMAPE'].append(smape(p_rs, DS['yte']))
    results['RS-SOFNN']['MASE'].append(mase(p_rs, DS['yte']))

    ap, _ = train_apso_fnn(DS['Xtt'], DS['ytt'], K0=20, n_iter=300, lr=0.01, N_win=10, seed=seed)
    p_ap  = ap.predict(DS['Xte'])
    r_ap  = rmse(p_ap, DS['yte'])
    results['APSO-FNN']['K'].append(int(ap.K))
    results['APSO-FNN']['RMSE'].append(r_ap)
    results['APSO-FNN']['sMAPE'].append(smape(p_ap, DS['yte']))
    results['APSO-FNN']['MASE'].append(mase(p_ap, DS['yte']))

    if r_dk < best_run_rmse:
        best_run_rmse = r_dk
        best_preds = {
            'DK-SOFNN': p_dk.copy(), 'GM-FTL':   p_gm.copy(),
            'RS-SOFNN': p_rs.copy(), 'APSO-FNN': p_ap.copy()
        }

    elapsed = time.time() - t0
    print(f"{run+1:>4}  {r_dk:>10.4f}  {r_gm:>10.4f}  {r_rs:>10.4f}  {r_ap:>10.4f}  {elapsed:>7.1f}s")

print('-' * len(hdr))
print(f'All {N_RUNS} runs complete.  Total: {(time.time()-t0)/60:.1f} min')
print(f'Mean RMSE — DK-SOFNN: {np.mean(results["DK-SOFNN"]["RMSE"]):.4f}  '
      f'GM-FTL: {np.mean(results["GM-FTL"]["RMSE"]):.4f}  '
      f'RS-SOFNN: {np.mean(results["RS-SOFNN"]["RMSE"]):.4f}  '
      f'APSO-FNN: {np.mean(results["APSO-FNN"]["RMSE"]):.4f}')

In [ ]:
# ── Cell 11: Fig 11 — Source FNN Structure Adjustment ─────────────
# K0=10 (blue solid) and K0=20 (red dashed)
# Y-axis: INTEGER ticks only  (no floats like 18.2, 18.4)

epochs_s = np.arange(len(rule_s10) - 1)   # 0..299

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(epochs_s, rule_s10[1:],
        color='royalblue', linewidth=1.8, label=f'Initial rules = 10',
        marker='o', markevery=30, markersize=4)
ax.plot(epochs_s, rule_s20[1:],
        color='red', linewidth=1.8, linestyle='--', label=f'Initial rules = 20',
        marker='s', markevery=30, markersize=4)

ax.axhline(y=int(rule_s10[-1]), color='royalblue', linestyle=':', alpha=0.6)
ax.axhline(y=int(rule_s20[-1]), color='red',       linestyle=':', alpha=0.6)

# Force integer Y-axis ticks
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

ax.set_xlabel('Iteration')
ax.set_ylabel('Number of Fuzzy Rules')
ax.set_title('Fig 11 — Structure Adjustment Process of the Source FNN\n'
             'Combined Cycle Power Plant Dataset', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True)

# Annotations: final rule count as INTEGER
ax.annotate(f'Converged → {int(rule_s10[-1])} rules',
            xy=(epochs_s[-1], int(rule_s10[-1])),
            xytext=(-90, 10), textcoords='offset points',
            fontsize=8, color='royalblue',
            arrowprops=dict(arrowstyle='->', color='royalblue'))
ax.annotate(f'Converged → {int(rule_s20[-1])} rules',
            xy=(epochs_s[-1], int(rule_s20[-1])),
            xytext=(-90, -20), textcoords='offset points',
            fontsize=8, color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.savefig('fig11_source_structure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig11_source_structure.png')
print(f'Source FNN: K0=10 → {int(rule_s10[-1])} rules,  K0=20 → {int(rule_s20[-1])} rules')

In [ ]:
# ── Cell 12: Fig 12 — Target FNN Structure Adjustment ─────────────
# DK-SOFNN starting from K0=15
# Y-axis: INTEGER ticks only

epochs_t = np.arange(len(rule_tgt) - 1)

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(epochs_t, rule_tgt[1:],
        color='seagreen', linewidth=1.8, label=f'DK-SOFNN (Initial rules = 15)',
        marker='^', markevery=30, markersize=4)

ax.axhline(y=int(rule_tgt[-1]), color='seagreen', linestyle=':', alpha=0.6)

# Force integer Y-axis ticks
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

ax.set_xlabel('Iteration')
ax.set_ylabel('Number of Fuzzy Rules')
ax.set_title('Fig 12 — Structure Adjustment Process of the Target FNN (DK-SOFNN)\n'
             'Combined Cycle Power Plant Dataset', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True)

ax.annotate(f'Converged → {int(rule_tgt[-1])} rules',
            xy=(epochs_t[-1], int(rule_tgt[-1])),
            xytext=(-120, 15), textcoords='offset points',
            fontsize=8, color='seagreen',
            arrowprops=dict(arrowstyle='->', color='seagreen'))

plt.tight_layout()
plt.savefig('fig12_target_structure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig12_target_structure.png')
print(f'Target FNN: K0=15 → converged to {int(rule_tgt[-1])} rules')

In [ ]:
# ── Cell 13: Fig 13 — Training RMSE (Different Fixed Structures) ──

epochs_tr = np.arange(1, 301)

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.plot(epochs_tr, rmse_11,  color='royalblue',  linewidth=1.5, label='Rules = 11')
ax.plot(epochs_tr, rmse_13,  color='darkorange', linewidth=1.5, label='Rules = 13', linestyle='--')
ax.plot(epochs_tr, rmse_15,  color='seagreen',   linewidth=1.5, label='Rules = 15', linestyle='-.')
ax.plot(epochs_tr, rmse_tgt, color='crimson',    linewidth=2.0, label='DK-SOFNN (adaptive)', linestyle=':')

ax.set_xlabel('Iteration (Epoch)')
ax.set_ylabel('Training RMSE (normalised)')
ax.set_title('Fig 13 — Training RMSE with Different Fixed Structures\n'
             'Combined Cycle Power Plant Dataset', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True)
ax.set_xlim([1, 300])

plt.tight_layout()
plt.savefig('fig13_training_rmse.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig13_training_rmse.png')

In [ ]:
# ── Cell 14: Fig 14 & 15 — Testing Values and Error (Fixed Structures) ─

N_SHOW   = 100
test_idx = np.arange(N_SHOW)
actual   = y_te[:N_SHOW]

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
fig.suptitle('Testing Performance with Different Fixed Structures\n'
             'Combined Cycle Power Plant Dataset',
             fontsize=12, fontweight='bold')

# ---- Fig 14: Predicted vs Actual ----
ax = axes[0]
ax.plot(test_idx, actual,          color='black',      lw=2.0, label='Actual', zorder=5)
ax.plot(test_idx, pred11[:N_SHOW], color='royalblue',  lw=1.2, label='K = 11', linestyle='--')
ax.plot(test_idx, pred13[:N_SHOW], color='darkorange', lw=1.2, label='K = 13', linestyle='-.')
ax.plot(test_idx, pred15[:N_SHOW], color='seagreen',   lw=1.2, label='K = 15', linestyle=':')
ax.set_ylabel('Output (normalised EP)')
ax.set_title('Fig 14 — Testing Values with Different Fixed Structures')
ax.legend(fontsize=9)
ax.grid(True)

# ---- Fig 15: Absolute Error ----
ax = axes[1]
ax.plot(test_idx, np.abs(pred11[:N_SHOW] - actual), color='royalblue',  lw=1.2, label='K = 11')
ax.plot(test_idx, np.abs(pred13[:N_SHOW] - actual), color='darkorange', lw=1.2, label='K = 13', linestyle='--')
ax.plot(test_idx, np.abs(pred15[:N_SHOW] - actual), color='seagreen',   lw=1.2, label='K = 15', linestyle='-.')
ax.set_xlabel('Test Sample Index')
ax.set_ylabel('Absolute Error (normalised)')
ax.set_title('Fig 15 — Testing Error with Different Fixed Structures')
ax.legend(fontsize=9)
ax.grid(True)

plt.tight_layout()
plt.savefig('fig14_15_test_structures.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig14_15_test_structures.png')

In [ ]:
# ── Cell 15: Fig 16 & 17 — Method Comparison ─────────────────────

N_SHOW   = 100
test_idx = np.arange(N_SHOW)
actual   = y_te[:N_SHOW]

m_colors = {'DK-SOFNN': 'crimson',    'GM-FTL':   'royalblue',
            'RS-SOFNN': 'darkorange', 'APSO-FNN': 'seagreen'}
m_styles = {'DK-SOFNN': '-',          'GM-FTL':   '--',
            'RS-SOFNN': '-.',          'APSO-FNN': ':'}

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
fig.suptitle('Method Comparison — Combined Cycle Power Plant Dataset',
             fontsize=12, fontweight='bold')

# ---- Fig 16: Predicted vs Actual ----
ax = axes[0]
ax.plot(test_idx, actual, color='black', lw=2.2, label='Actual', zorder=5)
for method, preds in best_preds.items():
    ax.plot(test_idx, preds[:N_SHOW],
            color=m_colors[method], lw=1.4,
            linestyle=m_styles[method], label=method)
ax.set_ylabel('Output (normalised EP)')
ax.set_title('Fig 16 — Testing Values: DK-SOFNN vs Other Methods')
ax.legend(fontsize=9)
ax.grid(True)

# ---- Fig 17: Absolute Error ----
ax = axes[1]
for method, preds in best_preds.items():
    ax.plot(test_idx, np.abs(preds[:N_SHOW] - actual),
            color=m_colors[method], lw=1.4,
            linestyle=m_styles[method], label=method)
ax.set_xlabel('Test Sample Index')
ax.set_ylabel('Absolute Error (normalised)')
ax.set_title('Fig 17 — Testing Error: DK-SOFNN vs Other Methods')
ax.legend(fontsize=9)
ax.grid(True)

plt.tight_layout()
plt.savefig('fig16_17_method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig16_17_method_comparison.png')

In [ ]:
# ── Cell 16: Table III — Performance Summary (30 runs) ───────────

yrange = DS['ymax'] - DS['ymin']

print('\n' + '='*80)
print('Table III — CCPP Dataset Performance (mean ± std over 30 independent runs)')
print('='*80)
print(f"{'Method':<12}  {'Rules':>14}  {'RMSE (norm)':>18}  "
      f"{'sMAPE':>16}  {'MASE':>16}")
print('-'*80)

for method in ['DK-SOFNN', 'GM-FTL', 'RS-SOFNN', 'APSO-FNN']:
    r  = results[method]
    km = np.mean(r['K']);    ks = np.std(r['K'])
    rm = np.mean(r['RMSE']); rs_ = np.std(r['RMSE'])
    sm = np.mean(r['sMAPE']); ss = np.std(r['sMAPE'])
    mm = np.mean(r['MASE']);  ms = np.std(r['MASE'])

    # Rules shown as rounded integers (no 18.2 / 18.4 floats)
    k_str = f"{int(round(km))} ± {int(round(ks))}"

    print(f"{method:<12}  {k_str:>14}  "
          f"{rm:.4f} ± {rs_:.4f}  "
          f"{sm:.4f} ± {ss:.4f}  "
          f"{mm:.4f} ± {ms:.4f}")

print('='*80)
print(f'Output range: {DS["ymin"]:.1f} – {DS["ymax"]:.1f} MW  (span ≈ {yrange:.1f} MW)')
print(f'DK-SOFNN RMSE in MW ≈ {np.mean(results["DK-SOFNN"]["RMSE"])*yrange:.2f} MW')

In [ ]:
# ── Cell 17: Pandas Table Display ────────────────────────────────

rows = []
for method in ['DK-SOFNN', 'GM-FTL', 'RS-SOFNN', 'APSO-FNN']:
    r  = results[method]
    rows.append({
        'Method':       method,
        'Rules':        f"{int(round(np.mean(r['K'])))} ± {int(round(np.std(r['K'])))}",
        'RMSE (norm)':  f"{np.mean(r['RMSE']):.4f} ± {np.std(r['RMSE']):.4f}",
        'RMSE (MW)':    f"{np.mean(r['RMSE'])*yrange:.3f} ± {np.std(r['RMSE'])*yrange:.3f}",
        'sMAPE':        f"{np.mean(r['sMAPE']):.4f} ± {np.std(r['sMAPE']):.4f}",
        'MASE':         f"{np.mean(r['MASE']):.4f} ± {np.std(r['MASE']):.4f}",
    })

df_table = pd.DataFrame(rows).set_index('Method')
print('\n📊 Table III Equivalent — CCPP Dataset')
display(df_table)

In [ ]:
# ── Cell 18: Print All Fuzzy Rules ────────────────────────────────
# Shows: Initial rules → Rules pruned (removed) → Rules added → Final rules
# Feature names: AT=Ambient Temperature, V=Vacuum, AP=Ambient Pressure, RH=Relative Humidity

FEATURE_NAMES = ['AT', 'V', 'AP', 'RH']

def center_label(v):
    if   v < 0.20: return 'Very Low'
    elif v < 0.40: return 'Low'
    elif v < 0.60: return 'Medium'
    elif v < 0.80: return 'High'
    else:          return 'Very High'

def width_label(v):
    if   v < 0.15: return 'Narrow'
    elif v < 0.35: return 'Moderate'
    else:          return 'Wide'

def print_rule(num, C_row, S_row, W_val, tag=''):
    ante = []
    for j, fn in enumerate(FEATURE_NAMES):
        ante.append(f"{fn} is {center_label(C_row[j])} [{width_label(S_row[j])} spread]"
                    f" (c={C_row[j]:.3f}, σ={S_row[j]:.3f})")
    ante_str = "\n             AND ".join(ante)
    tendency = 'High Net Power' if W_val > 0 else 'Low Net Power'
    print(f"  Rule {num:>2}{tag}:")
    print(f"    IF   {ante_str}")
    print(f"    THEN {tendency}   [weight = {W_val:+.4f}]")
    print()


# ─────────────────────────────────────────────────────────────────
# (A) INITIAL SOURCE RULES  (K0 = 20, before any pruning/adding)
# ─────────────────────────────────────────────────────────────────
print('=' * 70)
print(f'(A) INITIAL SOURCE RULES  [K0 = {len(init20)} rules — before structure adjustment]')
print('=' * 70)
for entry in init20:
    print_rule(entry['rule'], entry['C'], entry['S'], entry['W'])


# ─────────────────────────────────────────────────────────────────
# (B) PRUNED (REMOVED) RULES during Source FNN training
# ─────────────────────────────────────────────────────────────────
print('=' * 70)
print(f'(B) RULES PRUNED from Source FNN (K0=20)  [{len(pruned20)} rules removed]')
print('=' * 70)
if pruned20:
    for i, entry in enumerate(pruned20, 1):
        print(f'  Pruned at Epoch {entry["epoch"]} (was Rule #{entry["rule_idx"]})')
        print_rule(i, entry['C'], entry['S'], entry['W'], tag=' [REMOVED]')
else:
    print('  (No rules were pruned during source training)\n')


# ─────────────────────────────────────────────────────────────────
# (C) ADDED RULES during Source FNN training
# ─────────────────────────────────────────────────────────────────
print('=' * 70)
print(f'(C) RULES ADDED to Source FNN (K0=20)  [{len(added20)} rules added]')
print('=' * 70)
if added20:
    for i, entry in enumerate(added20, 1):
        print(f'  Added at Epoch {entry["epoch"]} (new Rule #{entry["rule_idx"]+1})')
        # Show center only (S not stored at add time)
        c_str = ', '.join([f'{FEATURE_NAMES[j]}={entry["C"][j]:.3f}' for j in range(len(entry['C']))])
        print(f'    Center: [{c_str}]  weight ≈ {entry["W"]:+.4f}\n')
else:
    print('  (No rules were added during source training)\n')


# ─────────────────────────────────────────────────────────────────
# (D) FINAL SOURCE RULES (after 300 epochs)
# ─────────────────────────────────────────────────────────────────
print('=' * 70)
print(f'(D) FINAL SOURCE FNN RULES  [K = {int(src20.K)} rules after training]')
print('=' * 70)
for k in range(src20.K):
    print_rule(k+1, src20.C[k], src20.S[k], src20.W[k])


# ─────────────────────────────────────────────────────────────────
# (E) FINAL TARGET (DK-SOFNN) RULES (after 300 epochs)
# ─────────────────────────────────────────────────────────────────
print('=' * 70)
print(f'(E) FINAL TARGET FNN (DK-SOFNN) RULES  [K = {int(dk_fnn.K)} rules after training]')
print(f'    (Started from K0=15, transferred from source, then adapted)')
print('=' * 70)
for k in range(dk_fnn.K):
    print_rule(k+1, dk_fnn.C[k], dk_fnn.S[k], dk_fnn.W[k])


# ─────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────
print('=' * 70)
print('RULE SUMMARY')
print('=' * 70)
print(f'  Source FNN   : started with K0=20 rules')
print(f'                 rules pruned  = {len(pruned20)}')
print(f'                 rules added   = {len(added20)}')
print(f'                 final rules   = {int(src20.K)}')
print(f'  Source FNN   : started with K0=10 rules')
print(f'                 final rules   = {int(src10.K)}')
print(f'  Target FNN   : started with K0=15 rules')
print(f'                 final rules   = {int(dk_fnn.K)}')
print('='*70)